# Can we find a snake's head and tail?

The species classifier might do better if it also saw a close crop of the snake's head (the scale pattern around the
eyes and snout is a classic clue) or its tail. We have no head or tail labels, so before building anything we run a
small test: ask Grounding DINO tiny (the text-prompted detector from the mask notebook) for "snake head" and "snake
tail" on 50 photos, and judge on a contact sheet whether each box lands on the right body
part.

A part passes if at least 40 of the 50 boxes are right. "Right" means the box covers the head (or tail) and little
else. The photos may not be redistributed, so the contact sheets with the boxes drawn are saved outside the repo and
this notebook shows counts only. It needs the clean photos and the experiment subset, found by name with `kagglehub`.
It runs on Kaggle with a GPU and internet on, in a few minutes.

In [1]:
import hashlib, json, os, sys, time, warnings
from collections import Counter
from pathlib import Path

os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1")
warnings.filterwarnings("ignore")

import kagglehub, pandas as pd, torch, transformers
from PIL import Image, ImageDraw

SRC = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(SRC if SRC.exists() else Path("../src").resolve()))
from serpentguard import video

ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working/heads") if ON_KAGGLE else Path("../work/heads")
SHEETS = OUT / "sheets"
SHEETS.mkdir(parents=True, exist_ok=True)

N_PHOTOS = 50
SEED = 42
DETECTOR_ID = "IDEA-Research/grounding-dino-tiny"
PROMPT = "snake head . snake tail ."
PARTS = ["head", "tail"]
BOX_THRESHOLD = TEXT_THRESHOLD = 0.25
PASS_AT = 40
TILE, COLUMNS = 320, 10
BOX_COLOUR = (235, 40, 40)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| transformers", transformers.__version__)

device: cuda | transformers 5.20.2


## 1. The 50 photos

The same experiment subset as the other notebooks (its fingerprint is checked). We take its training photos and draw
50 with `round_robin`, which takes the species in turn so that rare species are not drowned out by common ones.

In [2]:
def dataset(slug):
    return Path(kagglehub.dataset_download(f"puneetbajajiitm/{slug}"))

DATA = dataset("serpentguard-indian-snakes-clean")
subset = pd.read_csv(dataset("serpentguard-m3-subset") / "subset.csv", dtype={"observer_id": str})
fingerprint = hashlib.sha256("\n".join(sorted(subset["path"])).encode()).hexdigest()
assert fingerprint.startswith("40196d19a68e"), "this is not the experiment subset"

subset_train = subset[subset["split"] == "train"].reset_index(drop=True)
photos = video.round_robin(subset_train, N_PHOTOS, seed=SEED)
print(f"{len(subset_train):,} training photos in the subset; drew {len(photos)} from {photos['species'].nunique()} species")

13,925 training photos in the subset; drew 50 from 50 species


## 2. Ask Grounding DINO for both parts in one pass

One text prompt names both parts, `snake head . snake tail .`, so each photo goes through the detector once. Every box
comes back with the phrase it matched. For each photo and each part we keep the box with the highest score, or record
"none" if the detector drew no box for that part at the 0.25 thresholds. A box whose matched phrase names both parts
counts for both, so the same box can appear as head and as tail; the printout below shows how often that happens.

In [3]:
processor = transformers.AutoProcessor.from_pretrained(DETECTOR_ID)
detector = transformers.AutoModelForZeroShotObjectDetection.from_pretrained(DETECTOR_ID).to(DEVICE).eval()


phrases = Counter()  # which phrase each raw box matched, for the record


@torch.inference_mode()
def best_boxes(image):
    """{part: (box, score)} with the best-scoring box per part; a part with no box is left out."""
    inputs = processor(images=image, text=PROMPT, return_tensors="pt").to(DEVICE)
    outputs = detector(**inputs)
    found = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]])[0]
    labels = found.get("text_labels", found.get("labels"))
    best = {}
    for label, score, box in zip(labels, found["scores"].tolist(), found["boxes"].tolist()):
        phrases[label] += 1
        for part in PARTS:
            if part in label and score > best.get(part, (None, -1.0))[1]:
                best[part] = ([round(v, 1) for v in box], round(score, 4))
    return best


started = time.perf_counter()
rows = []
for path in photos["path"]:
    image = Image.open(DATA / "data" / path).convert("RGB")
    best = best_boxes(image)
    for part in PARTS:
        box, score = best.get(part, (None, None))
        rows.append(dict(path=path, part=part, box=json.dumps(box) if box else "none", score=score, right=None))
pilot = pd.DataFrame(rows)
print("phrases the raw boxes matched:", dict(phrases))
print(f"{len(photos)} photos in {time.perf_counter() - started:.0f} s")
for part in PARTS:
    found = pilot[(pilot["part"] == part) & (pilot["box"] != "none")]
    print(f"{part}: a box on {len(found)} of {len(photos)} photos, median score "
          f"{found['score'].median():.2f}" if len(found) else f"{part}: no boxes")

phrases the raw boxes matched: {'snake head tail': 8, 'snake': 1, 'tail': 31, 'head tail': 9, 'head': 17, 'snake head snake': 5, 'snake head snake tail': 24, 'snake head': 13, 'snake tail': 4, 'snake snake tail': 2}
50 photos in 31 s
head: a box on 47 of 50 photos, median score 0.44
tail: a box on 48 of 50 photos, median score 0.43


## 3. Contact sheets

One sheet per part: the 50 photos as thumbnails with the chosen box in red and the photo's number in the corner. The
numbers are the row order of `pilot.csv` for that part. These sheets show the photos, so they stay outside the repo and
are not displayed here; each box is judged on its sheet, and the
verdicts go into `pilot.csv`.

In [4]:
def tile(path, box, number):
    image = Image.open(DATA / "data" / path).convert("RGB")
    scale = TILE / max(image.size)
    canvas = Image.new("RGB", (TILE, TILE), (30, 30, 30))
    small = image.resize((round(image.width * scale), round(image.height * scale)))
    canvas.paste(small, ((TILE - small.width) // 2, (TILE - small.height) // 2))
    draw = ImageDraw.Draw(canvas)
    if box != "none":
        x1, y1, x2, y2 = (v * scale + off for v, off in zip(json.loads(box), [(TILE - small.width) // 2, (TILE - small.height) // 2] * 2))
        draw.rectangle([x1, y1, x2, y2], outline=BOX_COLOUR, width=3)
    label = f"{number}" if box != "none" else f"{number} none"
    draw.rectangle([0, 0, 8 * len(label) + 10, 18], fill=(0, 0, 0))
    draw.text((4, 3), label, fill=(255, 255, 255))
    return canvas


for part in PARTS:
    part_rows = pilot[pilot["part"] == part].reset_index(drop=True)
    rows_needed = -(-len(part_rows) // COLUMNS)
    sheet = Image.new("RGB", (COLUMNS * TILE, rows_needed * TILE))
    for i, row in part_rows.iterrows():
        sheet.paste(tile(row["path"], row["box"], i), ((i % COLUMNS) * TILE, (i // COLUMNS) * TILE))
    sheet.save(SHEETS / f"sheet_{part}.png")
print("sheets:", sorted(p.name for p in SHEETS.iterdir()))

sheets: ['sheet_head.png', 'sheet_tail.png']


## 4. Save the table

`right` is left empty here; it is filled in from the verdicts on the sheets. A part passes with at least 40 of 50 right.

In [5]:
pilot.to_csv(OUT / "pilot.csv", index=False)
print(pilot.groupby("part")["box"].agg(photos="size", boxes=lambda s: int((s != "none").sum())))

      photos  boxes
part               
head      50     47
tail      50     48


## 5. A fair retry: one prompt per pass, and two more detectors

The first try gave the detector both phrases at once, and it often matched the joint phrase "snake head snake tail",
which boxes the whole snake. Here each part gets its own pass. We try Grounding DINO tiny and base with a separate
prompt per part ("snake head ." then "snake tail ."), and OWLv2 (a different kind of open-vocabulary detector) with
the queries "a snake's head" and "a snake's tail" and a score threshold of 0.1. The best box per photo and part is
kept, as before; the same 50 photos and the same pass mark (40 of 50 right).

In [6]:
DINO_MODELS = {"dino_tiny": "IDEA-Research/grounding-dino-tiny", "dino_base": "IDEA-Research/grounding-dino-base"}
DINO_PROMPTS = {"head": "snake head .", "tail": "snake tail ."}
OWL_ID = "google/owlv2-base-patch16-ensemble"
OWL_QUERIES = {"head": "a snake's head", "tail": "a snake's tail"}
OWL_THRESHOLD = 0.1
RETRY_SHEETS = SHEETS / "retry"
RETRY_SHEETS.mkdir(parents=True, exist_ok=True)
images = {path: Image.open(DATA / "data" / path).convert("RGB") for path in photos["path"]}


def fmt(box, score):
    return (json.dumps([round(v, 1) for v in box]), round(score, 4)) if box is not None else ("none", None)


@torch.inference_mode()
def run_dino(model_id):
    proc = transformers.AutoProcessor.from_pretrained(model_id)
    model = transformers.AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(DEVICE).eval()
    out = []
    for path, image in images.items():
        for part, prompt in DINO_PROMPTS.items():
            inputs = proc(images=image, text=prompt, return_tensors="pt").to(DEVICE)
            found = proc.post_process_grounded_object_detection(
                model(**inputs), inputs.input_ids, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
                target_sizes=[image.size[::-1]])[0]
            scores = found["scores"].tolist()
            top = max(range(len(scores)), key=scores.__getitem__) if scores else None
            out.append((path, part) + fmt(found["boxes"][top].tolist() if top is not None else None,
                                          scores[top] if top is not None else 0))
    del model
    torch.cuda.empty_cache()
    return out


@torch.inference_mode()
def run_owl():
    proc = transformers.Owlv2Processor.from_pretrained(OWL_ID)
    model = transformers.Owlv2ForObjectDetection.from_pretrained(OWL_ID).to(DEVICE).eval()
    parts = list(OWL_QUERIES)
    out = []
    for path, image in images.items():
        inputs = proc(text=[list(OWL_QUERIES.values())], images=image, return_tensors="pt").to(DEVICE)
        outputs = model(**inputs)
        side = max(image.size)  # OWLv2 pads the photo to a square, so boxes are fractions of the square
        scores = outputs.logits[0].sigmoid()  # one score per candidate box and query
        centres = outputs.pred_boxes[0].float().cpu() * side  # (cx, cy, w, h)
        for k, part in enumerate(parts):
            top = int(scores[:, k].argmax())
            score = float(scores[top, k])
            cx, cy, w, h = centres[top].tolist()
            box = [max(cx - w / 2, 0), max(cy - h / 2, 0), min(cx + w / 2, image.width), min(cy + h / 2, image.height)]
            out.append((path, part) + (fmt(box, score) if score >= OWL_THRESHOLD else fmt(None, 0)))
    del model
    torch.cuda.empty_cache()
    return out


def save_sheets(model, rows):
    frame = pd.DataFrame(rows, columns=["path", "part", "box", "score"])
    for part, group in frame.groupby("part"):
        group = group.reset_index(drop=True)
        sheet = Image.new("RGB", (COLUMNS * TILE, -(-len(group) // COLUMNS) * TILE))
        for i, row in group.iterrows():
            sheet.paste(tile(row["path"], row["box"], i), ((i % COLUMNS) * TILE, (i // COLUMNS) * TILE))
        sheet.save(RETRY_SHEETS / f"sheet_{model}_{part}.png")
    frame.insert(0, "model", model)
    frame.assign(right=None).to_csv(OUT / f"pilot_retry_{model}.csv", index=False)  # saved as soon as the model is done
    return frame


RUNNERS = {**{name: (lambda model_id=model_id: run_dino(model_id)) for name, model_id in DINO_MODELS.items()},
           "owlv2": run_owl}
done, failed = [], {}
for name, runner in RUNNERS.items():
    try:
        done.append(save_sheets(name, runner()))
    except Exception as error:  # one model failing must not lose the others
        failed[name] = f"{type(error).__name__}: {error}"[:300]
        torch.cuda.empty_cache()
failures = pd.DataFrame([dict(model=n, path="", part="", box=f"error: {m}", score=None) for n, m in failed.items()])
retry = pd.concat(done + [failures], ignore_index=True).assign(right=None)
retry.to_csv(OUT / "pilot_retry.csv", index=False)
print("failed models:", failed or "none")
print(retry[retry["part"] != ""].groupby(["model", "part"])["box"].agg(photos="size", boxes=lambda s: int((s != "none").sum())))
print("retry sheets:", sorted(p.name for p in RETRY_SHEETS.iterdir()))

failed models: none
                photos  boxes
model     part               
dino_base head      50     50
          tail      50     50
dino_tiny head      50     49
          tail      50     49
owlv2     head      50     42
          tail      50     45
retry sheets: ['sheet_dino_base_head.png', 'sheet_dino_base_tail.png', 'sheet_dino_tiny_head.png', 'sheet_dino_tiny_tail.png', 'sheet_owlv2_head.png', 'sheet_owlv2_tail.png']


## 6. Verdicts

Each box was judged on its contact sheet: right when it covers the head (or tail) and little else, wrong
otherwise (a photo with no box counts as wrong). The verdicts are in the `right` column of
`results/m4/heads/pilot.csv` (section 2's single pass) and `results/m4/heads/pilot_retry.csv` (the retry),
and the counts below are tallied from them. A part passes with at least 40 of 50.

| model | prompting | part | boxes drawn | right of 50 |
|---|---|---|---|---|
| Grounding DINO tiny | both prompts in one pass (section 2) | head | 47 | 4 |
| Grounding DINO tiny | both prompts in one pass (section 2) | tail | 48 | 1 |
| Grounding DINO tiny | one prompt per pass | head | 49 | 4 |
| Grounding DINO tiny | one prompt per pass | tail | 49 | 1 |
| Grounding DINO base | one prompt per pass | head | 50 | 3 |
| Grounding DINO base | one prompt per pass | tail | 50 | 0 |
| OWLv2 | one query per part | head | 42 | 4 |
| OWLv2 | one query per part | tail | 45 | 3 |

No model comes near 40 of 50 for either part. Separate prompts did not help the Grounding DINO models: they still
box the whole snake, and the head and tail boxes are mostly the same box. OWLv2 more often leaves a photo without a
box, and its few right boxes are the same close-ups of a head, or a tail hanging free. The sheets stay outside the repo.